In [1]:
using LinearAlgebra

In [2]:
function compute_nodes(Nx,Ny)
    del_x = 1/(Nx-1)
    del_y = 1/(Ny-1)
    return del_x,del_y
end

compute_nodes (generic function with 1 method)

In [3]:
function int_mat(Nx,Ny)
    u_ini = zeros(Ny,Nx)
    v_ini = zeros(Ny,Nx)
    phi_ini = zeros(Ny,Nx)
    ome_ini = zeros(Ny,Nx)
    return u_ini,v_ini,phi_ini,ome_ini

end

int_mat (generic function with 1 method)

In [4]:
function boundary_condition( u_ini , v_ini , phi_ini , ome_ini )

    u_ini[:,Ny] .= 0
    v_ini[:,Ny] .= 0
    phi_ini[:,Ny] .= 0
    # down boundary
    u_ini[:,1] .= 0
    v_ini[:,1] .= 0
    phi_ini[:,1] .= 0
    # left boundary
    u_ini[1,:] .= 0
    v_ini[1,:] .= 0
    phi_ini[1,:] .= 0
    # right boundary
    u_ini[Nx,:] .= 1
    v_ini[Nx,:] .= 0
    phi_ini[Nx,:] .= 0

    # Votex boundary

    # down boundary
    ome_ini[1,:] .= (2 .* phi_ini[2,:]) ./ (del_y^2)
    # left boundary
    ome_ini[:,1] .= (2 .* phi_ini[:,2]) ./ (del_x^2)
    # right boundary
    ome_ini[:,Nx] .= ( 2 .*phi_ini[:,Nx-1]) ./ (del_x^2)
    # up boundary
    ome_ini[Ny,:] .= ( 2 .* phi_ini[Ny-1,:] .+ 2 * del_y) ./ (del_y^2)

    return u_ini , v_ini , phi_ini , ome_ini 
    
end

boundary_condition (generic function with 1 method)

In [5]:
function compute_ome( u_ini , v_ini , phi_ini , ome_ini , del_t , Re )
    ome = zeros( Ny , Nx )
    for i = 2 : Nx-1
        for j = 2 : Ny-1
            ome[i,j] = ome_ini[i,j] + del_t * (1/Re * ( ( ( ome_ini[i+1,j] - 2 * ome_ini[i,j] + ome_ini[i-1,j] ) / (del_x^2) ) + ( ome_ini[i,j+1] - 2*ome_ini[i,j] + ome_ini[i,j-1] ) / (del_y^2) ) -( 
            u_ini[i,j] * ( ( ome_ini[i+1,j]-ome_ini[i-1,j] ) / (2 * del_x) )+ v_ini[i,j] * ( ( ome_ini[i,j+1] - ome_ini[i,j-1] ) / (2 * del_y) ) )  )
        end
    end
    ome[1,:] = ome_ini[1,:]
    ome[:,1] = ome_ini[:,1]
    ome[:,Nx] = ome_ini[:,Nx]
    ome[Ny,:] = ome_ini[Ny,:]
    return ome
end

compute_ome (generic function with 1 method)

In [6]:
function compute_phi(phi_ini , ome , del_x , c , mode)
    phi_inf = phi_ini
    phi = zeros(Nx,Ny)
    if mode ==1 
        for i = 2 : Nx-1

            for j = 2 : Ny-1
                
                phi[i,j] = 1/4 * (phi_inf[i+1,j] + phi_inf[i-1,j] + phi_inf[i,j+1] + phi_inf[i,j-1]-(del_x^2) * ome[i,j])
            
            end

        end

        while abs(norm(phi,1) - norm(phi_inf,1)) > 10^-6

            phi_inf=phi
            phi = zeros(Nx,Ny)

            for i = 2 : Nx-1

                for j = 2 : Ny-1

                    phi[i,j] = 1/4 * (phi_inf[i+1,j] + phi_inf[i-1,j] + phi_inf[i,j+1] + phi_inf[i,j-1]-(del_x^2) * ome[i,j])

                end
            end
        end
        
    end
    # if mode == 2 
    #     A = zeros(Nx-2 * Ny-2 , Nx-2 * Ny-2)
    #     B = zeros(Nx-2 * Ny-2 , Nx-2 * Ny-2)

    #     for i = 1 : Nx - 2

    #         for j = 1 : Ny - 2

    #             if i == 1
                
    #             A[101 * (i - 1) + j ,101 * (i - 1) + j] = 1
    #             A[101 * (i - 1) + j ,101 * (i - 2) + j] = -c/4 
    #             A[101 * (i - 1) + j ,101 * (i - 1) + j - 1] = -c/4
    #             B[101 * (i - 1) + j ,101 * (i + 1 - 1) + j] = c/4
    #             B[101 * (i - 1) + j ,101 * (i - 1) + j + 1] = c/4 
    #             B[101 * (i - 1) + j ,101 * (i - 1) + j ] = 1 - c
                
    #         end

    #     end

    #     Y = zeros(Nx - 2 * Ny - 2 , 1)

    #     for i = 1 : Nx - 2

    #         for j = 1 : Nx - 2 

    #             Y[101 * (i - 1) + j ]=ome[i,j]
                
    #         end

    #     end

    #     while abs(norm(phi,1) - norm(phi_inf,1)) > 10^-6
    #         phi_inf=phi
    #         X = zeros(Nx * Ny , 1)
    #         for i = 1 : Nx

    #             for j = 1 : Nx

    #                 X[101 * (i - 1) + j ]=phi_inf[i,j]
    #                 Y[101 * (i - 1) + j ]=ome[i,j]
                    
    #             end

    #         end

    #         C = - (del_x^2) * c / 4 * Y .* I(Nx * Ny)

    #         phi = A^-1 * B + A^-1 * C 
            
    #     end
    # end
         return phi
end

compute_phi (generic function with 1 method)

In [7]:
function compute_speed(phi,del_x)
    u = zeros(Nx,Ny)
    v = zeros(Nx,Ny)
    for i = 2: Ny-1
        for j = 2 : Nx-1
            u[i,j] = (phi[i,j+1] - phi[i,j-1])/(2 * del_x)
            v[i,j] = (phi[i+1,j] - phi[i-1,j])/(2 * del_x)
        end
    end

    u[Nx,:] .= 1

    return u,v
end

compute_speed (generic function with 1 method)

In [8]:
del_t = 0.001
Re = 400
Nx = 51
Ny = 51
del_x,del_y = compute_nodes(Nx,Ny)
u_ini,v_ini,phi_ini,ome_ini = int_mat(Nx,Ny)
u_ini,v_ini,phi_ini,ome_ini = boundary_condition(u_ini,v_ini,phi_ini,ome_ini)
ome = compute_ome(u_ini,v_ini,phi_ini,ome_ini,del_t,Re)
phi = compute_phi(phi_ini , ome , del_x , 1.4 , 1)
u,v = compute_speed(phi,del_x)
u_ini = u
v_ini = v
phi_ini = phi
ome_ini = ome
u = zeros(Ny,Nx)
v = zeros(Ny,Nx)
phi = zeros(Ny,Nx)
ome = zeros(Ny,Nx)
while abs(norm(u,1)-norm(u_ini,1)) > 10^-6 || abs(norm(v,1)-norm(v_ini,1)) > 10^-6
    u_ini = u
    v_ini = v
    phi_ini = phi
    ome_ini = ome
    u_ini,v_ini,phi_ini,ome_ini = boundary_condition(u_ini,v_ini,phi_ini,ome_ini)
    ome = compute_ome(u_ini,v_ini,phi_ini,ome_ini,del_t,Re)
    phi = compute_phi(phi_ini , ome , del_x , 1.4 , 1)
    u,v = compute_speed(phi,del_x)
    @show abs(norm(u,1)-norm(u_ini,1)) , abs(norm(v,1)-norm(v_ini,1))
end

(abs(norm(u, 1) - norm(u_ini, 1)), abs(norm(v, 1) - norm(v_ini, 1))) = (0.3790008440937669, 0.778585410513265)
(abs(norm(u, 1) - norm(u_ini, 1)), abs(norm(v, 1) - norm(v_ini, 1))) = (0.3760101029939662, 0.7675839945481937)
(abs(norm(u, 1) - norm(u_ini, 1)), abs(norm(v, 1) - norm(v_ini, 1))) = (0.371564238729583, 0.756670814314772)
(abs(norm(u, 1) - norm(u_ini, 1)), abs(norm(v, 1) - norm(v_ini, 1))) = (0.36719198615062254, 0.7459425146037466)
(abs(norm(u, 1) - norm(u_ini, 1)), abs(norm(v, 1) - norm(v_ini, 1))) = (0.36290238571098854, 0.7353966898707727)
(abs(norm(u, 1) - norm(u_ini, 1)), abs(norm(v, 1) - norm(v_ini, 1))) = (0.35868489585010366, 0.725029682842349)
(abs(norm(u, 1) - norm(u_ini, 1)), abs(norm(v, 1) - norm(v_ini, 1))) = (0.3545410472756956, 0.7148386474810753)
(abs(norm(u, 1) - norm(u_ini, 1)), abs(norm(v, 1) - norm(v_ini, 1))) = (0.35046638928458407, 0.7048203748651023)
(abs(norm(u, 1) - norm(u_ini, 1)), abs(norm(v, 1) - norm(v_ini, 1))) = (0.34646860879720265, 0.694972473

Excessive output truncated after 524317 bytes.

(abs(norm(u, 1) - norm(u_ini, 1)), abs(norm(v, 1) - norm(v_ini, 1))) = 

In [1]:
u

UndefVarError: UndefVarError: `u` not defined